# 🏡 UK Real Estate Dashboard – Cloud Demo Version

This notebook launches the full **Real Estate + Macroeconomics Predictive Dashboard** using Streamlit + Cloudflared from Colab or remote environments.

## 🔍 What It Does
- Launches a Streamlit app (via background shell)
- Tunnels a public URL using `cloudflared`
- Loads model + data from **Google Cloud Storage**
- Runs real-time predictions, batch uploads, SHAP explanations, and economic scenario simulations

## 🧠 Why This Is Useful
This notebook is ideal for:
- **Sharing a live app demo** to recruiters without deploying on Streamlit Cloud
- **Testing the app pipeline** end-to-end (from GCS to prediction)
- **Running from any environment** (Colab, Jupyter, VM...)

## ⚙️ How To Use
- Run all cells
- Wait ~15s for the link to appear
- Click the public Cloudflare URL to access the dashboard!

Enjoy! 🌍


In [ ]:
!pip install streamlit

!pip uninstall -y numpy
!pip install numpy==1.24

# Chạy streamlit ngầm (log sẽ được ghi vào file)
!streamlit run streamlit_macro_dashboard.py --server.port 8502 > log.txt 2>&1 &
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

# Đợi 5 giây cho streamlit khởi động
import time; time.sleep(10)

# Mở tunnel và tạo link public
!./cloudflared tunnel --url http://localhost:8502


In [ ]:
%%writefile streamlit_macro_dashboard.py

# 📊 Real Estate + Macroeconomics Dashboard – Final Version with Predict + SHAP + Batch + Compare + WOW Effects

import streamlit as st
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from google.cloud import storage
import os, json, joblib, numpy as np
from catboost import CatBoostRegressor, Pool
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error
import shap

# 🎨 Must be FIRST
st.set_page_config(layout="wide")
st.title("\U0001F3E1 Real Estate Price & Macroeconomic Dashboard")
st.caption("\U0001F680 Built for Data Scientists, Economic Strategists & Hiring Managers")

# === GCP Setup ===
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "gcp-key.json"
bucket = "boothill2001-dataset"
data_path = "uk_property_data/processed/merged_real_estate_macro.parquet"
feat_path = "uk_property_data/models/feature_names.json"
model_gcs_paths = {
    "CatBoost": "uk_property_data/models/CatBoost_model.cbm",
    "XGBoost": "uk_property_data/models/XGBoost_model.json",
    "LightGBM": "uk_property_data/models/LightGBM_model.pkl"
}
model_paths = {
    "CatBoost": "models/CatBoost_model.cbm",
    "XGBoost": "models/XGBoost_model.json",
    "LightGBM": "models/LightGBM_model.pkl"
}

def download_model_if_missing(gcs_path, local_path):
    if not os.path.exists(local_path):
        client = storage.Client()
        blob = client.bucket(bucket).blob(gcs_path)
        os.makedirs(os.path.dirname(local_path), exist_ok=True)
        blob.download_to_filename(local_path)
        st.info(f"📦 Downloaded model from GCS → {local_path}")

@st.cache_resource
def load_data():
    local_file = "merged_real_estate_macro.parquet"
    if not os.path.exists(local_file):
        client = storage.Client()
        blob = client.bucket(bucket).blob(data_path)
        blob.download_to_filename(local_file)
    return pd.read_parquet(local_file)

@st.cache_data
def load_feature_names():
    local = "models/feature_names.json"
    if not os.path.exists(local):
        os.makedirs("models", exist_ok=True)
        blob = storage.Client().bucket(bucket).blob(feat_path)
        blob.download_to_filename(local)
    with open(local, "r") as f:
        return json.load(f)

df = load_data()
feature_names = load_feature_names()

# === SIDEBAR ===
st.sidebar.header("\U0001F3AF Filters & Config")
years = st.sidebar.multiselect("Chọn năm", sorted(df['Year'].unique()), default=[2020,2021,2022])
macro_cols = [col for col in df.columns if col not in df.select_dtypes(include='object').columns and col not in ['Log_Price','Price'] and df[col].nunique() > 20]
macro_col = st.sidebar.selectbox("\U0001F4C8 Chỉ số kinh tế", macro_cols)

# === 📊 CHARTS ===
st.subheader("\U0001F4C8 Price & Macro Trend Over Time")
df_filtered = df[df["Year"].isin(years)]
grouped = df_filtered.groupby(["Year", "Month"])[["Log_Price", macro_col]].mean()
for col in ['Year','Month']:
    if col in grouped.columns:
        grouped = grouped.drop(columns=col)
grouped = grouped.reset_index()
grouped["Date"] = pd.to_datetime(grouped[['Year','Month']].assign(DAY=1))

fig, ax1 = plt.subplots(figsize=(12, 5))
ax1.plot(grouped["Date"], grouped["Log_Price"], label="Log Price", color="blue")
ax2 = ax1.twinx()
ax2.plot(grouped["Date"], grouped[macro_col], label=macro_col, color="red")
st.pyplot(fig)

# === 🔍 Correlation ===
st.subheader("\U0001F50D Tương quan giữa Log_Price và các biến kinh tế")
corr = df[macro_cols + ["Log_Price"]].corr()
st.dataframe(corr.style.background_gradient(cmap="coolwarm", axis=None))

# === 📊 Distributions ===
st.subheader("\U0001F4CA Phân phối giá và macro")
c1, c2 = st.columns(2)
with c1:
    st.markdown("**Phân phối Log_Price**")
    sns.histplot(df['Log_Price'], kde=True); st.pyplot(plt.gcf()); plt.clf()
with c2:
    st.markdown(f"**Phân phối {macro_col}**")
    sns.histplot(df[macro_col], kde=True); st.pyplot(plt.gcf()); plt.clf()

# === 🧪 Scenario Simulation ===
st.subheader("\U0001F9EA Giả lập tình huống kinh tế")
sim_df = df.copy()
user_inputs = {}
for macro in macro_cols[:3]:
    val = st.slider(f"{macro}", float(df[macro].min()), float(df[macro].max()), float(df[macro].mean()), step=0.1)
    sim_df[macro] = val
    user_inputs[macro] = val
st.metric("\U0001F4A1 Giá Log_Price trung bình (giả lập)", f"{sim_df['Log_Price'].mean():.2f}")

# === 🔢 PREDICTION ===
st.subheader("\U0001F52E Dự đoán Log_Price theo chỉ số vĩ mô tùy chỉnh")
model_choice = st.selectbox("Chọn mô hình", list(model_paths.keys()))
model_path = model_paths[model_choice]
download_model_if_missing(model_gcs_paths[model_choice], model_path)

input_df = pd.DataFrame([0]*len(feature_names), index=feature_names).T
for k,v in user_inputs.items():
    if k in input_df.columns: input_df[k] = v

try:
    if model_choice == "CatBoost":
        model = CatBoostRegressor(); model.load_model(model_path)
        pred = model.predict(Pool(input_df, feature_names=feature_names))[0]
    elif model_choice == "XGBoost":
        model = xgb.Booster(); model.load_model(model_path)
        pred = model.predict(xgb.DMatrix(input_df.values, feature_names=feature_names))[0]
    else:
        model = joblib.load(model_path)
        pred = model.predict(input_df)[0]
    st.success(f"\U0001F4CC Dự đoán Log_Price: {pred:.2f}")
except Exception as e:
    st.error(f"❌ Lỗi: {e}")

# === 📅 Batch Prediction ===
st.subheader("\U0001F4C5 Dự đoán hàng loạt từ CSV (Batch Prediction)")
uploaded = st.file_uploader("Tải file CSV", type=["csv"])
if uploaded:
    batch = pd.read_csv(uploaded)
    for col in feature_names:
        if col not in batch.columns: batch[col] = 0
    batch = batch[feature_names]
    if model_choice == "XGBoost":
        preds = model.predict(xgb.DMatrix(batch.values, feature_names=feature_names))
    else:
        preds = model.predict(batch)
    st.dataframe(pd.DataFrame({"Prediction": preds}))

# === 📊 Model Comparison ===
st.subheader("\U0001F4CA So sánh mô hình với tập kiểm tra nhỏ (500 mẫu)")
x_test = df.sample(500, random_state=42)
y_test = x_test["Log_Price"]
for col in feature_names:
    if col not in x_test.columns:
        x_test[col] = 0
x_test = x_test[feature_names]

scores = {}
for name, path in model_paths.items():
    try:
        download_model_if_missing(model_gcs_paths[name], path)
        if name == "CatBoost":
            m = CatBoostRegressor(); m.load_model(path)
            y_pred = m.predict(Pool(x_test, feature_names=feature_names))
        elif name == "XGBoost":
            booster = xgb.Booster(); booster.load_model(path)
            y_pred = booster.predict(xgb.DMatrix(x_test.values, feature_names=feature_names))
        else:
            m = joblib.load(path)
            y_pred = m.predict(x_test)
        scores[name] = {"MAE": mean_absolute_error(y_test, y_pred), "RMSE": np.sqrt(mean_squared_error(y_test, y_pred))}
    except Exception as e:
        st.warning(f"⚠️ {name} failed: {e}")

st.dataframe(pd.DataFrame(scores).T.style.background_gradient(cmap="YlGn"))

# === 🧠 SHAP Explainability ===
st.subheader("\U0001F9E0 Giải thích mô hình bằng SHAP (Explainable AI)")
if model_choice == "CatBoost":
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(input_df)
    shap.summary_plot(shap_values, input_df, plot_type="bar")
    st.pyplot(bbox_inches="tight")